# Breakout
**Experiments:** DQN and PPO  
**Resume:** automatically loads the latest checkpoint  
**Storage:** only the newest checkpoint is kept

## 1. Setup
Install the required Atari and reinforcement learning packages.

In [ ]:
%pip install -q "gymnasium[atari,accept-rom-license]>=1.0" "stable-baselines3[extra]>=2.4" "ale-py>=0.10" "autorom[accept-rom-license]" "opencv-python-headless" "imageio[ffmpeg]" "pandas" "matplotlib"
!AutoROM --accept-license >/dev/null 2>&1 || true

## 2. Configuration
Set the seed, folders, evaluation episodes, and device.

In [ ]:
import re
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

import torch as th
import torch.nn as nn

from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.utils import set_random_seed

SEED = 42
EVAL_EPISODES = 20

random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
set_random_seed(SEED)

DEVICE = "cuda" if th.cuda.is_available() else "cpu"

ROOT = Path("as2_breakout")
MODEL_DIR = ROOT / "models"
CHECKPOINT_DIR = ROOT / "checkpoints"

for folder in [MODEL_DIR, CHECKPOINT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

## 3. Training Target
Change only `TOTAL_TIMESTEPS` if the trained agent needs more learning.  
The notebook will resume from the latest checkpoint instead of starting again.

In [ ]:
TOTAL_TIMESTEPS = 1_000_000
CHECKPOINT_INTERVAL_STEPS = 250_000

## 4. Environment
Breakout uses image observations and discrete actions. The objective is to maximise cumulative reward.

In [ ]:
ENV_ID = "ALE/Breakout-v5"

def make_breakout_env(seed=SEED):
    def _init():
        env = gym.make(
            ENV_ID,
            frameskip=1,
            repeat_action_probability=0.0,
        )

        env = AtariWrapper(
            env,
            screen_size=84,
            frame_skip=4,
            terminal_on_life_loss=False,
            clip_reward=False,
        )

        env = Monitor(env)
        env.reset(seed=seed)
        return env

    return _init


def build_env(seed=SEED):
    env = DummyVecEnv([make_breakout_env(seed)])
    return VecFrameStack(env, n_stack=4)


print("Using:", ENV_ID)

## 5. Custom CNN
The same custom CNN feature extractor is used for DQN and PPO.

In [ ]:
class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=512):
        super().__init__(observation_space, features_dim)

        channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(channels, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )

        with th.no_grad():
            sample = th.as_tensor(
                observation_space.sample()[None]
            ).float()
            n_flatten = self.cnn(sample).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU(),
        )

    def forward(self, observations):
        return self.linear(self.cnn(observations))


POLICY_KWARGS = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=512),
)

## 6. Utilities
Save a checkpoint every 250,000 steps and delete the previous checkpoint after the new one is safely written.

In [ ]:
class EpisodeRewardCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.rewards = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.rewards.append(
                    float(info["episode"]["r"])
                )
        return True


def latest_step_checkpoint(folder, prefix):
    files = list(
        Path(folder).glob(
            f"{prefix}_*_steps.zip"
        )
    )

    if not files:
        return None, 0

    def get_steps(path):
        match = re.search(
            r"_(\d+)_steps\.zip$",
            path.name,
        )
        return int(match.group(1)) if match else -1

    latest = max(files, key=get_steps)
    return latest, get_steps(latest)


class LatestStepCheckpointCallback(BaseCallback):
    def __init__(
        self,
        checkpoint_dir,
        prefix,
        interval,
        save_replay_buffer=False,
        verbose=1,
    ):
        super().__init__(verbose)

        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.prefix = prefix
        self.interval = interval
        self.save_replay_buffer = save_replay_buffer
        self.next_checkpoint = None

    def _on_training_start(self):
        current = int(self.model.num_timesteps)

        self.next_checkpoint = (
            (current // self.interval) + 1
        ) * self.interval

    def _remove_old_checkpoints(self, keep_base):
        for path in self.checkpoint_dir.glob(
            f"{self.prefix}_*_steps*"
        ):
            if not path.name.startswith(keep_base.name):
                path.unlink(missing_ok=True)

    def save_latest(self):
        steps = int(self.model.num_timesteps)

        base = (
            self.checkpoint_dir
            / f"{self.prefix}_{steps}_steps"
        )

        # Save the new checkpoint first.
        self.model.save(base)

        if (
            self.save_replay_buffer
            and hasattr(
                self.model,
                "save_replay_buffer",
            )
        ):
            self.model.save_replay_buffer(
                str(base) + "_replay_buffer.pkl"
            )

        # Remove older checkpoints only after saving succeeds.
        self._remove_old_checkpoints(base)

        if self.verbose:
            print(
                f"Latest checkpoint saved at "
                f"{steps:,} steps"
            )

    def _on_step(self):
        current = int(self.model.num_timesteps)

        if current >= self.next_checkpoint:
            self.save_latest()

            while self.next_checkpoint <= current:
                self.next_checkpoint += self.interval

        return True


def evaluate_model(
    model,
    env,
    episodes=EVAL_EPISODES,
):
    rewards, lengths = evaluate_policy(
        model,
        env,
        n_eval_episodes=episodes,
        deterministic=True,
        return_episode_rewards=True,
        warn=False,
    )

    return {
        "mean_reward": float(np.mean(rewards)),
        "std_reward": float(np.std(rewards)),
        "median_reward": float(np.median(rewards)),
        "mean_length": float(np.mean(lengths)),
    }


def plot_rewards(rewards, title, window=50):
    series = pd.Series(rewards, dtype=float)

    plt.figure(figsize=(9, 4))
    plt.plot(
        series,
        alpha=0.35,
        label="Episode reward",
    )

    if len(series) >= window:
        plt.plot(
            series.rolling(window).mean(),
            label=f"{window}-episode mean",
        )

    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

## 7. Random Baseline
A random policy provides a baseline before training.

In [ ]:
def random_baseline(
    episodes=EVAL_EPISODES,
):
    env = build_env(SEED + 99)
    returns = []

    obs = env.reset()
    total = 0.0

    while len(returns) < episodes:
        action = np.array(
            [env.action_space.sample()]
        )

        obs, reward, done, info = env.step(
            action
        )

        total += float(reward[0])

        if bool(done[0]):
            returns.append(total)
            total = 0.0

    env.close()

    return {
        "mean_reward": float(np.mean(returns)),
        "std_reward": float(np.std(returns)),
        "median_reward": float(np.median(returns)),
    }


random_result = random_baseline()
random_result

## 8. Experiment 1 — DQN
DQN automatically resumes from the newest checkpoint and continues until `TOTAL_TIMESTEPS`.

In [ ]:
dqn_env = build_env(SEED)
dqn_rewards = EpisodeRewardCallback()

dqn_dir = CHECKPOINT_DIR / "dqn"
dqn_ckpt, completed_steps = latest_step_checkpoint(
    dqn_dir,
    "breakout_dqn",
)

if dqn_ckpt is None:
    print("No DQN checkpoint found. Starting from 0.")

    dqn = DQN(
        "CnnPolicy",
        dqn_env,
        learning_rate=1e-4,
        buffer_size=100_000,
        learning_starts=20_000,
        batch_size=32,
        gamma=0.99,
        train_freq=4,
        gradient_steps=1,
        target_update_interval=10_000,
        exploration_fraction=0.20,
        exploration_initial_eps=1.0,
        exploration_final_eps=0.05,
        policy_kwargs=POLICY_KWARGS,
        verbose=1,
        seed=SEED,
        device=DEVICE,
    )

else:
    print("Resuming DQN from:", dqn_ckpt)
    print("Completed steps:", completed_steps)

    dqn = DQN.load(
        dqn_ckpt,
        env=dqn_env,
        device=DEVICE,
    )

    replay_path = dqn_ckpt.with_name(
        dqn_ckpt.stem
        + "_replay_buffer.pkl"
    )

    if replay_path.exists():
        dqn.load_replay_buffer(replay_path)
        print("Replay buffer loaded.")

dqn_checkpoint = LatestStepCheckpointCallback(
    checkpoint_dir=dqn_dir,
    prefix="breakout_dqn",
    interval=CHECKPOINT_INTERVAL_STEPS,
    save_replay_buffer=True,
)

remaining_steps = max(
    0,
    TOTAL_TIMESTEPS
    - int(dqn.num_timesteps),
)

print("Target steps:", TOTAL_TIMESTEPS)
print("Current steps:", dqn.num_timesteps)
print("Remaining steps:", remaining_steps)

if remaining_steps > 0:
    start = time.time()

    dqn.learn(
        total_timesteps=remaining_steps,
        reset_num_timesteps=False,
        callback=[
            dqn_rewards,
            dqn_checkpoint,
        ],
        progress_bar=True,
    )

    # Keep the exact newest training state for the next run.
    dqn_checkpoint.save_latest()

    print(
        f"Training time this run: "
        f"{(time.time() - start)/60:.1f} min"
    )

dqn.save(
    MODEL_DIR / "breakout_dqn_final"
)

### DQN Evaluation
Evaluate the current DQN model on separate episodes.

In [ ]:
plot_rewards(
    dqn_rewards.rewards,
    "Breakout DQN",
)

dqn_eval_env = build_env(
    SEED + 1000
)

dqn_result = evaluate_model(
    dqn,
    dqn_eval_env,
)

dqn_result

## 9. Experiment 2 — PPO
PPO also resumes automatically and keeps only the newest checkpoint.

In [ ]:
ppo_env = build_env(SEED + 1)
ppo_rewards = EpisodeRewardCallback()

ppo_dir = CHECKPOINT_DIR / "ppo"
ppo_ckpt, completed_steps = latest_step_checkpoint(
    ppo_dir,
    "breakout_ppo",
)

if ppo_ckpt is None:
    print("No PPO checkpoint found. Starting from 0.")

    ppo = PPO(
        "CnnPolicy",
        ppo_env,
        learning_rate=2.5e-4,
        n_steps=128,
        batch_size=256,
        n_epochs=4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.1,
        ent_coef=0.01,
        policy_kwargs=POLICY_KWARGS,
        verbose=1,
        seed=SEED,
        device=DEVICE,
    )

else:
    print("Resuming PPO from:", ppo_ckpt)
    print("Completed steps:", completed_steps)

    ppo = PPO.load(
        ppo_ckpt,
        env=ppo_env,
        device=DEVICE,
    )

ppo_checkpoint = LatestStepCheckpointCallback(
    checkpoint_dir=ppo_dir,
    prefix="breakout_ppo",
    interval=CHECKPOINT_INTERVAL_STEPS,
    save_replay_buffer=False,
)

remaining_steps = max(
    0,
    TOTAL_TIMESTEPS
    - int(ppo.num_timesteps),
)

print("Target steps:", TOTAL_TIMESTEPS)
print("Current steps:", ppo.num_timesteps)
print("Remaining steps:", remaining_steps)

if remaining_steps > 0:
    start = time.time()

    ppo.learn(
        total_timesteps=remaining_steps,
        reset_num_timesteps=False,
        callback=[
            ppo_rewards,
            ppo_checkpoint,
        ],
        progress_bar=True,
    )

    ppo_checkpoint.save_latest()

    print(
        f"Training time this run: "
        f"{(time.time() - start)/60:.1f} min"
    )

ppo.save(
    MODEL_DIR / "breakout_ppo_final"
)

### PPO Evaluation
Evaluate the current PPO model on separate episodes.

In [ ]:
plot_rewards(
    ppo_rewards.rewards,
    "Breakout PPO",
)

ppo_eval_env = build_env(
    SEED + 1001
)

ppo_result = evaluate_model(
    ppo,
    ppo_eval_env,
)

ppo_result

## 10. Results
Compare Random, DQN, and PPO.

In [ ]:
breakout_results = pd.DataFrame([
    {
        "Algorithm": "Random",
        **random_result,
    },
    {
        "Algorithm": "DQN",
        **dqn_result,
    },
    {
        "Algorithm": "PPO",
        **ppo_result,
    },
])

breakout_results